# SHL Hiring Assessment 2026 — HuBERT Segment Multimodal Candidate

**Fixed new candidate — no hyperparameter sweep.**

Current public champion to preserve separately: **0.3814**.

This candidate uses a materially different pretrained speech representation:
**HuBERT-base + beginning/middle/end 6-second segments + acoustic features + OOF ensemble**.

The notebook includes:
- 769 train / 216 test checks
- 16 kHz mono preprocessing
- HuBERT-base representation
- acoustic features
- leakage-safe 5-fold OOF modelling
- Ridge + ExtraTrees
- Ridge meta ensemble
- repeated-CV robustness diagnostic
- test-like validation diagnostic
- compulsory training RMSE
- required visualizations
- brief report
- final `submission.csv`

In [ ]:
import os, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import soundfile as sf
import librosa
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = "/kaggle/working"

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
BASE_DIR = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

train_df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))

assert len(train_df) == 769
assert len(test_df) == 216

y = train_df["label"].astype(float).to_numpy()

tr_paths = [os.path.join(TRAIN_DIR, f) for f in train_df["filename"]]
te_paths = [os.path.join(TEST_DIR, f) for f in test_df["filename"]]

assert all(os.path.exists(p) for p in tr_paths)
assert all(os.path.exists(p) for p in te_paths)

print("Train:", len(train_df), "Test:", len(test_df))
print("Label range:", y.min(), "to", y.max())

## Acoustic features

Compact duration, activity, spectral and MFCC statistics are used as an interpretable complementary branch.

In [ ]:
def acoustic_features(path, sr_target=16000):
    wav, sr = sf.read(path, dtype="float32")

    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    if sr != sr_target:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=sr_target)

    wav = np.nan_to_num(wav).astype("float32")

    duration = len(wav) / sr_target

    rms = librosa.feature.rms(
        y=wav, frame_length=1024, hop_length=512
    )[0]

    zcr = librosa.feature.zero_crossing_rate(
        wav, frame_length=1024, hop_length=512
    )[0]

    centroid = librosa.feature.spectral_centroid(
        y=wav, sr=sr_target, hop_length=512
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=wav, sr=sr_target, hop_length=512
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=wav, sr=sr_target, hop_length=512
    )[0]

    mfcc = librosa.feature.mfcc(
        y=wav, sr=sr_target, n_mfcc=13, hop_length=512
    )

    threshold = max(
        1e-5,
        0.02 * (np.max(np.abs(wav)) + 1e-12)
    )

    active_ratio = (
        np.abs(wav) > threshold
    ).mean()

    feats = [
        duration,
        active_ratio,
        rms.mean(), rms.std(), rms.min(), rms.max(),
        zcr.mean(), zcr.std(),
        centroid.mean(), centroid.std(),
        bandwidth.mean(), bandwidth.std(),
        rolloff.mean(), rolloff.std()
    ]

    for i in range(13):
        feats += [
            mfcc[i].mean(),
            mfcc[i].std()
        ]

    return np.asarray(feats, dtype="float32")


def build_acoustic(paths, cache):
    if os.path.exists(cache):
        print("Loading:", cache)
        return np.load(cache)

    X = np.vstack([
        acoustic_features(p)
        for p in tqdm(paths, desc="Acoustic")
    ]).astype("float32")

    np.save(cache, X)
    return X


Atr = build_acoustic(
    tr_paths,
    f"{WORK}/Atr_hubert_candidate.npy"
)

Ate = build_acoustic(
    te_paths,
    f"{WORK}/Ate_hubert_candidate.npy"
)

print("Acoustic train:", Atr.shape)
print("Acoustic test :", Ate.shape)

## HuBERT-base segment representation

Each recording uses three fixed 6-second views:
**beginning, middle and end**.

HuBERT-base has a 768-dimensional hidden representation. Mean + standard deviation pooling produces 1536 features per segment, giving **4608 features per recording**.

In [ ]:
from transformers import Wav2Vec2FeatureExtractor, HubertModel

MODEL_NAME = "facebook/hubert-base-ls960"
SR = 16000
CLIP_SEC = 6

processor = Wav2Vec2FeatureExtractor.from_pretrained(
    MODEL_NAME
)

hubert = HubertModel.from_pretrained(
    MODEL_NAME
).to(DEVICE).eval()

print("HuBERT loaded.")

In [ ]:
def extract_hubert(paths, cache):

    if os.path.exists(cache):
        print("Loading:", cache)
        return np.load(cache)

    features = []
    clip_len = SR * CLIP_SEC

    for path in tqdm(paths, desc="HuBERT"):

        wav, sr = sf.read(
            path,
            dtype="float32"
        )

        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        if sr != SR:
            wav = librosa.resample(
                wav,
                orig_sr=sr,
                target_sr=SR
            )

        wav = np.nan_to_num(
            wav
        ).astype("float32")

        n = len(wav)

        starts = [
            0,
            max(0, n // 2 - clip_len // 2),
            max(0, n - clip_len)
        ]

        clips = []

        for start in starts:

            x = wav[
                start:start + clip_len
            ]

            if len(x) < clip_len:
                x = np.pad(
                    x,
                    (0, clip_len - len(x))
                )

            clips.append(x)

        batch = processor(
            clips,
            sampling_rate=SR,
            return_tensors="pt",
            padding=True
        )

        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        with torch.inference_mode():

            h = hubert(
                **batch
            ).last_hidden_state

            emb = torch.cat(
                [
                    h.mean(dim=1),
                    h.std(dim=1)
                ],
                dim=1
            )

        features.append(
            emb.cpu().numpy().reshape(-1)
        )

    X = np.vstack(
        features
    ).astype("float32")

    np.save(cache, X)

    return X


Htr = extract_hubert(
    tr_paths,
    f"{WORK}/Htr_hubert_3x6.npy"
)

Hte = extract_hubert(
    te_paths,
    f"{WORK}/Hte_hubert_3x6.npy"
)

print("HuBERT train:", Htr.shape)
print("HuBERT test :", Hte.shape)

assert Htr.shape == (769, 4608)
assert Hte.shape == (216, 4608)

## Leakage-safe OOF base models

The target is continuous, so standard `KFold` is used.

Fixed base models:
1. HuBERT Ridge
2. HuBERT ExtraTrees
3. Acoustic Ridge
4. HuBERT + Acoustic Ridge

In [ ]:
def RMSE(a, b):
    return float(
        np.sqrt(mean_squared_error(a, b))
    )

def Pearson(a, b):
    return float(
        pearsonr(a, b)[0]
    )


kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

names = [
    "HuBERT_Ridge",
    "HuBERT_ExtraTrees",
    "Acoustic_Ridge",
    "Combined_Ridge"
]

oof = np.zeros(
    (769, 4),
    dtype="float32"
)

tefold = np.zeros(
    (216, 4, 5),
    dtype="float32"
)

X_combined = np.hstack([
    Htr,
    Atr
])

Z_combined = np.hstack([
    Hte,
    Ate
])

for fold, (tr, va) in enumerate(
    kf.split(Htr)
):

    m1 = make_pipeline(
        StandardScaler(),
        Ridge(alpha=300)
    )

    m1.fit(Htr[tr], y[tr])

    oof[va, 0] = m1.predict(Htr[va])
    tefold[:, 0, fold] = m1.predict(Hte)


    m2 = ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=4,
        max_features=0.35,
        n_jobs=-1,
        random_state=SEED
    )

    m2.fit(Htr[tr], y[tr])

    oof[va, 1] = m2.predict(Htr[va])
    tefold[:, 1, fold] = m2.predict(Hte)


    m3 = make_pipeline(
        StandardScaler(),
        Ridge(alpha=100)
    )

    m3.fit(Atr[tr], y[tr])

    oof[va, 2] = m3.predict(Atr[va])
    tefold[:, 2, fold] = m3.predict(Ate)


    m4 = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1000)
    )

    m4.fit(
        X_combined[tr],
        y[tr]
    )

    oof[va, 3] = m4.predict(
        X_combined[va]
    )

    tefold[:, 3, fold] = m4.predict(
        Z_combined
    )


    print(
        f"Fold {fold+1}/5 | "
        f"HuBERT RMSE={RMSE(y[va], oof[va,0]):.4f} | "
        f"Combined RMSE={RMSE(y[va], oof[va,3]):.4f}"
    )

In [ ]:
rows = []

for j, name in enumerate(names):

    pred = np.clip(
        oof[:, j],
        0,
        5
    )

    rows.append({
        "Model": name,
        "OOF_RMSE": RMSE(y, pred),
        "OOF_Pearson": Pearson(y, pred)
    })

results = pd.DataFrame(rows).sort_values(
    "OOF_RMSE"
)

display(results)

## Ridge meta-ensemble

In [ ]:
meta = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10)
)

meta.fit(
    oof,
    y
)

meta_oof = np.clip(
    meta.predict(oof),
    0,
    5
)

print(
    "META OOF RMSE:",
    RMSE(y, meta_oof)
)

print(
    "META OOF Pearson:",
    Pearson(y, meta_oof)
)

## Compulsory training RMSE

This is explicitly reported as a training metric and is **not** treated as validation performance.

In [ ]:
final_train_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1000)
)

final_train_model.fit(
    X_combined,
    y
)

train_pred = np.clip(
    final_train_model.predict(X_combined),
    0,
    5
)

training_rmse = RMSE(
    y,
    train_pred
)

training_pearson = Pearson(
    y,
    train_pred
)

print(
    "COMPULSORY TRAINING RMSE:",
    training_rmse
)

print(
    "TRAINING PEARSON:",
    training_pearson
)

## Repeated-CV robustness diagnostic

Three fixed seeds are evaluated. This is a robustness check, not a hyperparameter sweep.

In [ ]:
robust_rows = []

for seed in [7, 42, 123]:

    rkf = KFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    pred = np.zeros(
        len(y)
    )

    for tr, va in rkf.split(
        X_combined
    ):

        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=1000)
        )

        model.fit(
            X_combined[tr],
            y[tr]
        )

        pred[va] = model.predict(
            X_combined[va]
        )

    pred = np.clip(
        pred,
        0,
        5
    )

    robust_rows.append({
        "seed": seed,
        "RMSE": RMSE(y, pred),
        "Pearson": Pearson(y, pred)
    })

robust_results = pd.DataFrame(
    robust_rows
)

display(robust_results)

print(
    "Mean repeated-CV RMSE:",
    robust_results["RMSE"].mean()
)

print(
    "Mean repeated-CV Pearson:",
    robust_results["Pearson"].mean()
)

## Test-like validation diagnostic

Training records whose numeric filename stem is within 0–215 are evaluated as a robustness holdout.

This uses **no test labels** and does not copy labels to the evaluation set.

In [ ]:
def numeric_stem(s):
    try:
        return int(Path(str(s)).stem)
    except Exception:
        return None


ids = np.array([
    numeric_stem(x)
    for x in train_df["filename"]
])

mask = np.array([
    v is not None and 0 <= v <= 215
    for v in ids
])

print(
    "Test-like training records:",
    int(mask.sum())
)

if mask.sum() >= 30:

    tr_idx = np.where(~mask)[0]
    va_idx = np.where(mask)[0]

    diagnostic_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1000)
    )

    diagnostic_model.fit(
        X_combined[tr_idx],
        y[tr_idx]
    )

    diag_pred = np.clip(
        diagnostic_model.predict(
            X_combined[va_idx]
        ),
        0,
        5
    )

    print(
        "Test-like RMSE:",
        RMSE(y[va_idx], diag_pred)
    )

    print(
        "Test-like Pearson:",
        Pearson(y[va_idx], diag_pred)
    )
else:
    print(
        "Not enough test-like records."
    )

## Final test prediction and distribution checks

In [ ]:
test_base = tefold.mean(
    axis=2
)

final_pred = np.clip(
    meta.predict(test_base),
    0,
    5
)

print("Count:", len(final_pred))
print("Mean :", final_pred.mean())
print("Std  :", final_pred.std())
print("Min  :", final_pred.min())
print("Max  :", final_pred.max())

assert len(final_pred) == 216
assert np.isfinite(final_pred).all()
assert np.all(
    (final_pred >= 0) &
    (final_pred <= 5)
)

## Required visualizations

In [ ]:
# Training label distribution
plt.figure(figsize=(8,5))
plt.hist(y, bins=20)
plt.xlabel("Training grammar score")
plt.ylabel("Count")
plt.title("Training Label Distribution")
plt.tight_layout()
plt.show()

# OOF actual vs predicted
plt.figure(figsize=(7,6))
plt.scatter(y, meta_oof, alpha=0.6)
plt.xlabel("Actual score")
plt.ylabel("OOF predicted score")
plt.title("OOF Actual vs Predicted")
plt.tight_layout()
plt.show()

# Residual distribution
residuals = y - meta_oof
plt.figure(figsize=(8,5))
plt.hist(residuals, bins=30)
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("OOF Residual Distribution")
plt.tight_layout()
plt.show()

# RMSE comparison
plt.figure(figsize=(9,5))
plt.bar(results["Model"], results["OOF_RMSE"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("RMSE")
plt.title("OOF RMSE Comparison")
plt.tight_layout()
plt.show()

# Pearson comparison
plt.figure(figsize=(9,5))
plt.bar(results["Model"], results["OOF_Pearson"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Pearson")
plt.title("OOF Pearson Comparison")
plt.tight_layout()
plt.show()

# Test prediction distribution
plt.figure(figsize=(8,5))
plt.hist(final_pred, bins=25)
plt.xlabel("Predicted grammar score")
plt.ylabel("Count")
plt.title("Test Prediction Distribution")
plt.tight_layout()
plt.show()

# OOF prediction correlation matrix
corr = pd.DataFrame(
    oof,
    columns=names
).corr()

plt.figure(figsize=(7,6))
plt.imshow(
    corr.values,
    aspect="auto"
)
plt.xticks(
    range(len(names)),
    names,
    rotation=35,
    ha="right"
)
plt.yticks(
    range(len(names)),
    names
)
plt.colorbar(
    label="Correlation"
)
plt.title(
    "OOF Prediction Correlation Matrix"
)
plt.tight_layout()
plt.show()

## Final report

### Problem
Continuous regression from spoken-English audio to a 0–5 grammar score.

### Preprocessing
WAV files are loaded as mono and standardized to 16 kHz.

### Feature extraction
HuBERT-base is used as a frozen pretrained speech encoder. Beginning, middle, and end 6-second segments are represented using temporal mean and standard deviation pooling. Compact acoustic/MFCC/activity features provide a complementary branch.

### Modelling
Ridge and ExtraTrees base regressors generate leakage-safe OOF predictions. A regularized Ridge meta-model combines the OOF predictions.

### Validation
OOF RMSE and Pearson are reported, together with repeated-CV and test-like robustness diagnostics.

### Required training metric
Training RMSE is reported separately and is not presented as a validation score.

### Submission
Predictions are constrained to [0,5], checked for NaNs and test-order consistency, and saved as `submission.csv`.

### Limitation
The public leaderboard represents only part of the evaluation, so public score improvement does not guarantee hidden-test improvement.

## Final submission gate

- [x] 769 labelled training samples
- [x] 216 evaluation samples
- [x] continuous 0–5 output
- [x] train labels only
- [x] no same-ID label copying
- [x] leakage-safe OOF predictions
- [x] Pearson + RMSE
- [x] compulsory training RMSE
- [x] repeated-CV diagnostic
- [x] test-like diagnostic
- [x] prediction distribution
- [x] required visualizations
- [x] brief report
- [x] exact test filename order
- [x] final file named `submission.csv`

**Current champion 0.3814 must remain preserved separately.**

In [ ]:
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": final_pred
})

assert submission.shape == (216, 2)
assert submission["filename"].equals(
    test_df["filename"]
)
assert submission["label"].notna().all()
assert submission["label"].between(0,5).all()

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("=" * 60)
print("FINAL SUBMISSION CREATED")
print(submission_path)
print("=" * 60)

display(submission.head(10))
print(submission["label"].describe())